# 에이전트 메모리(Agent Memory)

이 노트북은 LLM agent가 왜 메모리(memory)를 필요로 하는지, 그리고 메모리를 어떤 종류로 나눠 설계하면 좋은지를 단계적으로 설명한다. 단순히 "기억한다"는 추상적인 표현이 아니라, 방금 대화한 내용을 잠시 들고 있는 short-term memory, 파일로 오래 남기는 long-term memory, 의미적으로 비슷한 과거 경험을 찾는 vector memory가 각각 어떤 역할을 맡는지 실험으로 확인한다.

## 학습 목표
- short-term, long-term, vector memory의 차이를 일상적인 비유와 함께 이해한다.
- memory retrieval이 document retrieval과 어떻게 결합되는지 설명할 수 있다.
- selective memory update가 왜 필요한지, 왜 모든 대화를 다 저장하면 안 되는지 이해한다.
- memory를 workflow에 연결했을 때 trace와 state에서 무엇을 확인해야 하는지 익힌다.

### 읽을 때 체크할 점
- 이 셀의 출력은 그 자체보다도 **어느 단계의 가정이 맞았는지/틀렸는지**를 보여주는 신호로 읽는 것이 좋다.
- 스터디나 면접에서는 `무엇을 했다`보다 `왜 이런 단계를 따로 분리했는가`를 설명할 수 있어야 한다.


## 개념 설명

이 첫 셀은 메모리 실험을 시작하기 전, 현재 노트북이 어느 Python 환경에서 실행 중인지 확인한다. memory notebook은 파일 기반 저장소(JSON)와 여러 helper 모듈을 함께 사용하므로, 경로와 커널이 조금만 어긋나도 "저장한 줄 알았는데 다른 곳에 저장"되는 문제가 생길 수 있다.

- **목적**: 노트북이 올바른 가상환경에서 실행 중인지 가장 먼저 확인한다.
- **핵심 로직**: 프로젝트 루트를 찾기 전에 현재 Python 실행 경로를 먼저 출력해 kernel 연결 상태를 점검한다.
- **주요 파라미터/변수**:
  - `sys.executable`: 현재 연결된 Jupyter kernel의 실제 Python 실행 파일 경로이다.

이 셀은 짧지만 중요하다. 메모리 시스템은 "어디에 저장했는가"가 핵심이므로, 실행 환경 확인이 곧 재현성 확인이다.

### 읽을 때 체크할 점
- 이 셀의 출력은 그 자체보다도 **어느 단계의 가정이 맞았는지/틀렸는지**를 보여주는 신호로 읽는 것이 좋다.
- 스터디나 면접에서는 `무엇을 했다`보다 `왜 이런 단계를 따로 분리했는가`를 설명할 수 있어야 한다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(sys.executable)

## 구현 준비

이 setup 셀은 메모리 실습에 필요한 핵심 모듈을 모두 불러오고, 장기 메모리 저장소 파일을 깨끗한 상태로 초기화한다. 노트북 안에 메모리 로직을 직접 쓰지 않고 `src/memory.py`에서 가져오는 이유는, 메모리 설계를 재사용 가능하고 inspectable하게 유지하기 위해서다.

- **목적**: short-term, long-term, vector memory 실험에 필요한 클래스와 helper를 준비한다.
- **핵심 로직**: 프로젝트 루트를 다시 한 번 안전하게 맞춘 뒤, `ShortTermMemory`, `LongTermMemory`, `VectorMemory`, `build_memory_augmented_context`, `selective_memory_update`, `run_workflow_with_memory` 등을 import한다. 또한 `agent_memory_notebook_store.json`이 이미 있으면 지워서 실험을 초기 상태에서 시작한다.
- **주요 파라미터/변수**:
  - `PROJECT_ROOT`: 메모리 파일과 `src/` 모듈을 찾기 위한 기준 경로이다.
  - `paths.logs_dir`: 장기 메모리 JSON을 저장할 디렉터리이다.
  - `memory_store_path`: 이번 노트북이 사용할 장기 메모리 파일 경로이다.

예를 들어 아래 코드에서:
- `if memory_store_path.exists(): memory_store_path.unlink()`: 이전 실행의 흔적을 지워, 이번 실험 결과만 보이게 한다.
- `pd.set_option('display.max_colwidth', 140)`: 메모리 텍스트가 길어도 표 안에서 잘리지 않게 한다.

학습용 노트북에서 setup을 명시적으로 적어두는 이유는, 메모리처럼 상태가 남는 기능일수록 "왜 이번에는 결과가 다르지?"라는 질문이 자주 나오기 때문이다.

### 읽을 때 체크할 점
- 이 셀의 출력은 그 자체보다도 **어느 단계의 가정이 맞았는지/틀렸는지**를 보여주는 신호로 읽는 것이 좋다.
- 스터디나 면접에서는 `무엇을 했다`보다 `왜 이런 단계를 따로 분리했는가`를 설명할 수 있어야 한다.


In [ ]:
import pandas as pd


from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

from src.config import get_paths
from src.ingestion import build_demo_index
from src.memory import (
    LongTermMemory,
    ShortTermMemory,
    VectorMemory,
    build_memory_augmented_context,
    display_memory,
    score_memory_importance,
    selective_memory_update,
    summarize_memory_events,
)
from src.utils import display_trace
from src.workflow import run_workflow_with_memory

pd.set_option('display.max_colwidth', 140)
paths = get_paths()
memory_store_path = paths.logs_dir / 'agent_memory_notebook_store.json'
if memory_store_path.exists():
    memory_store_path.unlink()


## 소개(Introduction)

agent에게 메모리가 필요한 이유는 컨텍스트 윈도(context window)가 무한하지 않기 때문이다. 모델은 한 번에 볼 수 있는 텍스트 양이 제한되어 있고, 이전 대화를 영원히 전부 들고 다니는 것은 비용과 노이즈 측면에서 모두 비효율적이다. 그래서 메모리를 종류별로 나눠 설계한다.

일상적인 비유로 보면 다음과 같다.
- **Short-term memory**: 방금 대화한 내용을 잠시 머릿속에 붙잡아 두는 작업 기억이다. 채팅 히스토리 버퍼와 비슷하다.
- **Long-term memory**: 오래 기억해야 할 사용자 선호나 사실을 메모장에 적어 두는 것과 같다. 이 프로젝트에서는 JSON 파일에 저장한다.
- **Vector memory**: 정확히 같은 문장이 아니어도 비슷한 기억을 떠올리는 방식이다. TF-IDF 기반 유사도 검색으로 구현한다.
- **Episodic memory**: "무슨 일이 언제 있었는가" 같은 사건 중심 기억이다.
- **Semantic memory**: 사건보다 더 일반화된 사실과 개념 지식이다.

왜 stateless pipeline과 memory agent가 다른가? stateless pipeline은 매 질문을 완전히 처음 보는 일회성 요청처럼 다룬다. 반면 memory agent는 이전 상호작용을 활용해 더 개인화되고 일관된 답을 만들 수 있다. 하지만 그만큼 잘못된 기억을 오래 끌고 갈 위험도 생긴다.

### 3종 메모리 비유
- **ShortTermMemory**: 방금 나눈 대화를 잠깐 잡아두는 버퍼
- **LongTermMemory**: 수첩에 적어 두는 오래가는 사실/선호
- **VectorMemory**: 정확한 표현은 달라도 비슷한 기억을 떠올리는 장치

왜 모든 대화를 다 기억하면 안 되는가:
- retrieval 노이즈가 늘어난다.
- 중요 정보가 덜 중요한 이벤트에 묻힌다.
- 저장 비용과 관리 복잡도가 커진다.


## 단기 메모리(Short-Term Memory)

short-term memory는 지금 대화에서 막 나온 정보를 잠깐 붙잡아 두는 버퍼다. 예를 들어 사용자가 "앞으로는 간결하게 답해줘"라고 말하면, 바로 다음 답변부터 그 스타일을 반영해야 한다. 하지만 이런 정보가 영구 저장될 필요는 없다.

이 셀은 가장 단순한 `ShortTermMemory` 클래스를 만들어 몇 개의 대화 이벤트를 쌓는다.

- **목적**: conversation buffer가 어떻게 동작하는지 가장 작은 예제로 확인한다.
- **핵심 로직**: `ShortTermMemory(max_items=6)`로 최대 이벤트 개수를 정한 뒤, `append_event(role, content)`로 사용자와 assistant의 대화 turn을 순서대로 추가한다.
- **주요 파라미터/변수**:
  - `max_items=6`: 버퍼가 너무 커지지 않도록 하는 상한이다.
  - `role`: `user`인지 `assistant`인지 나타내는 화자 정보이다.
  - `content`: 메모리에 저장할 실제 텍스트이다.

예를 들어 아래 코드에서:
- `append_event('user', 'Please keep future answers concise.')`: 앞으로 답변 스타일에 영향을 줄 수 있는 지시를 short-term memory에 남긴다.
- `short_term.to_frame()`: 메모리 내용을 표로 바꿔서 순서와 역할을 확인한다.

왜 이런 버퍼가 필요한가? reasoning chain이나 직전 turn을 계속 참조해야 하는 agent는 최근 맥락이 사라지면 바로 일관성을 잃는다.

### 실제 구현 펼쳐보기: `ShortTermMemory`
```python
class ShortTermMemory:
    def __init__(self, max_items: int | None = None) -> None:
        self.max_items = max_items
        self.events: list[dict[str, Any]] = []

    def append_event(
        self,
        kind: str,
        content: str,
        metadata: dict[str, Any] | None = None,
    ) -> dict[str, Any]:
        event = {
            "event_id": len(self.events) + 1,
            "kind": kind,
            "content": normalize_text(content),
            "metadata": metadata or {},
            "timestamp": iso_timestamp(),
        }
        self.events.append(event)
        if self.max_items is not None and len(self.events) > self.max_items:
            self.events = self.events[-self.max_items :]
        return event

    def last_n(self, limit: int) -> list[dict[str, Any]]:
        return self.events[-limit:]

    def clear(self) -> None:
        self.events.clear()

    def to_frame(self) -> pd.DataFrame:
        return pd.DataFrame(self.events, columns=["event_id", "kind", "content", "metadata", "timestamp"])
```

주목할 점:
- `append_event()`는 event_id와 timestamp를 붙여 저장한다.
- `max_items`가 있으면 슬라이딩 윈도우처럼 최근 것만 남긴다.
- `last_n()`은 컨텍스트 윈도 비용을 줄이기 위한 가장 단순한 선택 전략이다.


In [ ]:
short_term = ShortTermMemory(max_items=6)
short_term.append_event('user', 'Can you explain the rollout timeline?')
short_term.append_event('assistant', 'The pilot runs from March 10, 2025 to April 4, 2025.')
short_term.append_event('user', 'Please keep future answers concise.')
short_term.append_event('assistant', 'Noted. I will keep answers concise when possible.')
short_term.to_frame()


## 최근 메모리 다시 읽기

버퍼를 만든 뒤에는 그 안에서 필요한 부분만 다시 꺼내올 수 있어야 한다. 모든 과거 turn을 매번 전부 모델 입력에 넣으면 토큰 비용이 커지고, 오래된 문맥이 현재 질문을 오염시킬 수 있다. 그래서 보통 최근 N개만 추려 쓰는 방식이 자주 쓰인다.

- **목적**: short-term memory에서 최근 이벤트만 선택적으로 조회하는 방법을 확인한다.
- **핵심 로직**: `last_n(3)`이 가장 최근 3개의 메모리 이벤트를 반환하고, 이를 `DataFrame`으로 보여준다.
- **주요 파라미터/변수**:
  - `recent_turns`: 최근 메모리 슬라이스 결과이다.
  - `3`: 몇 개의 최근 이벤트를 가져올지 정하는 윈도 크기이다.

아래 코드에서 `short_term.last_n(3)`는 "지금 답하는 데 필요한 최신 문맥만 가져오자"는 뜻이다. 실제 서비스에서는 이 값이 너무 작으면 맥락을 놓치고, 너무 크면 노이즈가 쌓인다.

### 읽을 때 체크할 점
- 이 셀의 출력은 그 자체보다도 **어느 단계의 가정이 맞았는지/틀렸는지**를 보여주는 신호로 읽는 것이 좋다.
- 스터디나 면접에서는 `무엇을 했다`보다 `왜 이런 단계를 따로 분리했는가`를 설명할 수 있어야 한다.


In [ ]:
recent_turns = short_term.last_n(3)
pd.DataFrame(recent_turns)


### 장기 메모리(Long-Term Memory)

long-term memory는 사용자의 선호, 반복적으로 등장하는 사실, 장기간 유지해야 하는 규칙을 저장하는 공간이다. 일상적으로 비유하면 중요한 정보를 메모장이나 CRM에 적어두는 것과 비슷하다. short-term memory가 휘발성이라면, long-term memory는 세션이 끝나도 살아남는다.

이 셀은 JSON 파일 기반 `LongTermMemory` 저장소를 만들고, 사용자 선호와 팀 사실을 저장한 뒤, 다시 로드해 persistence를 확인한다.

- **목적**: 영속 저장이 되는 장기 메모리가 실제로 round-trip되는지 확인한다.
- **핵심 로직**: `store_memory(key, value, category=...)`로 항목을 저장하고, 새 인스턴스를 다시 만들어 같은 파일에서 메모리를 불러온다.
- **주요 파라미터/변수**:
  - `memory_store_path`: JSON 파일 저장 경로이다.
  - `key`: 나중에 다시 찾기 위한 메모리 식별자이다.
  - `category`: `preference`, `fact`처럼 메모리 성격을 구분하는 분류이다.

예를 들어:
- `mina_answer_style`: 사용자별 답변 스타일 선호를 저장한다.
- `apollo_pilot_city`: 나중에 다른 질문에서 참조할 수 있는 팀 사실 정보를 저장한다.

장기 메모리는 편리하지만, 잘못 저장된 사실이 오래 남는다는 점도 같이 기억해야 한다.

### 실제 구현 펼쳐보기: `LongTermMemory`
```python
class LongTermMemory:
    def __init__(self, path: Path) -> None:
        self.path = Path(path)
        self._items: list[dict[str, Any]] = self._load()

    def _load(self) -> list[dict[str, Any]]:
        if not self.path.exists():
            return []
        payload = read_json(self.path)
        return list(payload) if isinstance(payload, list) else []

    def _persist(self) -> None:
        ensure_directory(self.path.parent)
        write_json(self.path, self._items)

    def store_memory(
        self,
        key: str,
        value: str,
        category: str = "fact",
        metadata: dict[str, Any] | None = None,
        importance: float | None = None,
    ) -> dict[str, Any]:
        item = {
            "key": key,
            "value": normalize_text(value),
            "category": category,
            "metadata": metadata or {},
            "importance": importance,
            "updated_at": iso_timestamp(),
        }
        existing_index = next((index for index, entry in enumerate(self._items) if entry["key"] == key), None)
        if existing_index is None:
            self._items.append(item)
        else:
            self._items[existing_index] = item
        self._persist()
        return item

    def retrieve_memory(self, key: str) -> dict[str, Any] | None:
        return next((item for item in self._items if item["key"] == key), None)

    def list_items(self) -> list[dict[str, Any]]:
        return list(self._items)

    def search(self, query: str, limit: int = 3) -> list[dict[str, Any]]:
        query_tokens = content_tokens(query)
        matches: list[dict[str, Any]] = []
        for item in self._items:
            searchable_text = f"{item['key']} {item['value']} {item['category']}"
            lexical_score = overlap_ratio(query_tokens, content_tokens(searchable_text))
            exact_key_bonus = 0.25 if item["key"].lower() in normalize_text(query).lower() else 0.0
            score = round(min(1.0, lexical_score + exact_key_bonus), 3)
            if score <= 0.0:
                continue
            matches.append(
                {
                    "memory_type": "long_term",
                    "key": item["key"],
                    "text": item["value"],
                    "score": score,
                    "category": item["category"],
                    "metadata": item["metadata"],
                    "timestamp": item["updated_at"],
                }
            )
        return sorted(matches, key=lambda entry: entry["score"], reverse=True)[:limit]

    def to_frame(self) -> pd.DataFrame:
        return pd.DataFrame(self._items, columns=["key", "value", "category", "importance", "updated_at"])
```

핵심 해설:
- JSON 파일 기반이라 inspectable하고 로컬에서 재현 가능하다.
- `store_memory()`는 같은 key가 있으면 upsert한다.
- `search()`는 lexical overlap와 exact key bonus를 섞어 간단하지만 설명 가능한 검색을 한다.


In [ ]:
long_term = LongTermMemory(memory_store_path)
long_term.store_memory('mina_answer_style', 'Mina prefers concise answers that lead with exact dates.', category='preference')
long_term.store_memory('apollo_pilot_city', 'Team Apollo is piloting the scheduling template in Seoul.', category='fact')
reloaded_long_term = LongTermMemory(memory_store_path)
reloaded_long_term.to_frame()


### 저장한 장기 메모리 조회하기

메모리는 저장만큼 retrieval이 중요하다. 특히 long-term memory는 key-value 형태가 많기 때문에, 어떤 키를 주면 어떤 값이 나오는지 단순하고 예측 가능해야 한다. 복잡한 retrieval이 필요 없는 사실/선호라면 이렇게 직접 조회하는 방식이 가장 안정적이다.

- **목적**: 방금 저장한 장기 메모리가 원하는 키로 정확히 조회되는지 확인한다.
- **핵심 로직**: `retrieve_memory('mina_answer_style')`가 해당 key에 연결된 메모리 항목을 반환한다.
- **주요 파라미터/변수**:
  - `'mina_answer_style'`: 사용자 선호를 식별하는 메모리 키이다.
  - `value`: 실제 저장된 텍스트 값이다.

이 조회 결과가 중요한 이유는, personalization이 결국 이런 작은 규칙에서 시작되기 때문이다. 예를 들어 "답변은 날짜부터 시작하라" 같은 선호는 retrieval보다 direct lookup이 더 적합하다.

### 읽을 때 체크할 점
- 이 셀의 출력은 그 자체보다도 **어느 단계의 가정이 맞았는지/틀렸는지**를 보여주는 신호로 읽는 것이 좋다.
- 스터디나 면접에서는 `무엇을 했다`보다 `왜 이런 단계를 따로 분리했는가`를 설명할 수 있어야 한다.


In [ ]:
reloaded_long_term.retrieve_memory('mina_answer_style')


### 벡터 메모리(Vector Memory)

vector memory는 정확히 같은 표현이 아니어도 의미가 비슷한 과거 기억을 찾아오는 장치다. 사람으로 치면 "그 말과 비슷한 상황을 전에 어디서 들었지?"를 떠올리는 것과 비슷하다. 이 프로젝트에서는 외부 서비스 대신 TF-IDF 기반 cosine similarity로 단순하게 구현해, 내부 동작을 읽기 쉽게 유지한다.

- **목적**: 의미적으로 관련 있는 기억을 유사도 검색으로 찾아오는 과정을 본다.
- **핵심 로직**: `add_memory(text, metadata=..., importance=...)`로 메모리를 추가하고, `search(query, top_k=3)`로 질의와 가장 가까운 메모리를 찾는다.
- **주요 파라미터/변수**:
  - `text`: 벡터화할 메모리 문장이다.
  - `metadata`: 메모리 종류(`preference`, `fact`, `event`) 같은 부가 정보이다.
  - `importance`: 이후 업데이트/필터링에서 참고할 중요도 값이다.
  - `top_k=3`: 가장 관련 있는 메모리 3개만 가져온다.

예를 들어 아래 코드에서:
- `vector_memory.add_memory(...)`: 기억 문장을 인덱스에 넣는다.
- `vector_memory.search('How should I answer rollout timing for Mina?', top_k=3)`: 정확히 같은 문장이 아니어도, Mina와 rollout timing에 관련된 메모리를 상단으로 올린다.

결과 표에서는 `score`가 높은 메모리가 실제 질문 의도와 얼마나 잘 맞는지 확인해보자. Vector memory의 핵심은 "같은 키"가 아니라 "비슷한 의미"이다.

### 실제 구현 펼쳐보기: `VectorMemory`
```python
class VectorMemory:
    def __init__(self) -> None:
        self.entries: list[dict[str, Any]] = []
        self.vectorizer: TfidfVectorizer | None = None
        self.matrix: Any = None

    def _rebuild_index(self) -> None:
        if not self.entries:
            self.vectorizer = None
            self.matrix = None
            return
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
        self.matrix = self.vectorizer.fit_transform(entry["text"] for entry in self.entries)

    def add_memory(
        self,
        text: str,
        metadata: dict[str, Any] | None = None,
        importance: float = 0.5,
    ) -> dict[str, Any]:
        entry = {
            "memory_id": f"memory_{len(self.entries) + 1}",
            "text": normalize_text(text),
            "metadata": metadata or {},
            "importance": round(float(importance), 3),
            "timestamp": iso_timestamp(),
        }
        self.entries.append(entry)
        self._rebuild_index()
        return entry

    def search(self, query: str, top_k: int = 3, min_score: float = 0.0) -> list[dict[str, Any]]:
        if not self.entries or self.vectorizer is None or self.matrix is None:
            return []

        query_vector = self.vectorizer.transform([query])
        cosine_scores = cosine_similarity(query_vector, self.matrix).flatten()
        lexical_scores = [
            overlap_ratio(content_tokens(query), content_tokens(entry["text"]))
            for entry in self.entries
        ]

        ranked: list[dict[str, Any]] = []
        for index, entry in enumerate(self.entries):
            score = round(float((cosine_scores[index] * 0.85) + (lexical_scores[index] * 0.15)), 4)
            if score < min_score:
                continue
            ranked.append(
                {
                    "memory_type": "vector",
                    "memory_id": entry["memory_id"],
                    "text": entry["text"],
                    "score": score,
                    "importance": entry["importance"],
                    "metadata": entry["metadata"],
                    "timestamp": entry["timestamp"],
                }
            )

        return sorted(ranked, key=lambda entry: entry["score"], reverse=True)[:top_k]

    def to_frame(self) -> pd.DataFrame:
        return pd.DataFrame(self.entries, columns=["memory_id", "text", "importance", "metadata", "timestamp"])
```

읽는 포인트:
- 외부 vector DB 대신 TF-IDF + cosine으로 구현해 내부 동작을 드러낸다.
- `cosine 0.85 + lexical 0.15`로 의미 유사도와 표면 겹침을 함께 본다.
- 의미 기반 검색이지만 완전히 opaque하지 않다는 점이 교육용으로 중요하다.


In [ ]:
vector_memory = VectorMemory()
vector_memory.add_memory('Mina prefers concise rollout answers that begin with the exact launch date.', metadata={'kind': 'preference'}, importance=0.9)
vector_memory.add_memory('The rollout FAQ should mention the May 5, 2025 launch date.', metadata={'kind': 'fact'}, importance=0.8)
vector_memory.add_memory('The pilot retrospective is scheduled for June.', metadata={'kind': 'event'}, importance=0.5)
vector_search_results = vector_memory.search('How should I answer rollout timing for Mina?', top_k=3)
pd.DataFrame(vector_search_results)


## Agent에서의 메모리 검색(Memory Retrieval in Agents)

실제 agent는 문서만 검색하지 않고, 필요한 경우 메모리도 함께 검색해 context를 합친다. 흐름은 대체로 `query -> retrieve docs -> retrieve memories -> combine context` 순서다. 즉, 외부 근거와 내부 기억을 함께 본 뒤 답을 만드는 구조다.

- **목적**: 문서 retrieval과 memory retrieval이 하나의 context bundle로 결합되는 방식을 확인한다.
- **핵심 로직**: 먼저 `build_demo_index(persist=False)`로 문서 retriever를 만들고, `retriever.search(...)`로 관련 문서를 가져온 뒤, `build_memory_augmented_context(...)`로 문서와 메모리를 하나의 묶음으로 합친다.
- **주요 파라미터/변수**:
  - `retrieved_docs`: 문서 검색 결과이다.
  - `short_term_memory`, `long_term_memory`, `vector_memory`: 서로 다른 성격의 메모리 소스이다.
  - `top_k=3`: memory retrieval에서도 상위 몇 개를 합칠지 정한다.
  - `context_bundle['combined_context']`: 최종적으로 agent가 참고할 통합 문맥이다.

이 셀의 핵심은 문서와 메모리가 경쟁 관계가 아니라는 점이다. 문서는 사실 근거를 주고, 메모리는 사용자 맞춤 맥락을 준다. 둘이 섞이면 더 관련성 높은 답을 만들 수 있다. 단, 메모리가 문서 근거를 덮어쓰면 위험하므로 verifier와 함께 봐야 한다.

### 실제 구현 펼쳐보기: `retrieve_relevant_memories()`, `build_memory_augmented_context()`, `memory_results_to_docs()`
```python
def retrieve_relevant_memories(
    query: str,
    short_term_memory: ShortTermMemory | None = None,
    long_term_memory: LongTermMemory | None = None,
    vector_memory: VectorMemory | None = None,
    top_k: int = 3,
) -> list[dict[str, Any]]:
    matches: list[dict[str, Any]] = []
    query_tokens = content_tokens(query)

    if short_term_memory is not None:
        for event in short_term_memory.last_n(top_k * 2):
            score = round(min(1.0, overlap_ratio(query_tokens, content_tokens(event["content"])) + 0.2), 3)
            if score <= 0.0:
                continue
            matches.append(
                {
                    "memory_type": "short_term",
                    "key": f"event_{event['event_id']}",
                    "text": event["content"],
                    "score": score,
                    "category": event["kind"],
                    "metadata": event["metadata"],
                    "timestamp": event["timestamp"],
                }
            )

    if long_term_memory is not None:
        matches.extend(long_term_memory.search(query, limit=top_k))

    if vector_memory is not None:
        matches.extend(vector_memory.search(query, top_k=top_k))

    ranked = sorted(matches, key=lambda entry: (entry["score"], entry.get("importance", 0.0)), reverse=True)
    return ranked[:top_k]
```

```python
def build_memory_augmented_context(
    query: str,
    retrieved_docs: list[dict[str, Any]],
    short_term_memory: ShortTermMemory | None = None,
    long_term_memory: LongTermMemory | None = None,
    vector_memory: VectorMemory | None = None,
    top_k: int = 3,
) -> dict[str, Any]:
    memories = retrieve_relevant_memories(
        query=query,
        short_term_memory=short_term_memory,
        long_term_memory=long_term_memory,
        vector_memory=vector_memory,
        top_k=top_k,
    )
    combined_context = {
        "query": query,
        "documents": [doc["text"] for doc in retrieved_docs],
        "memories": [memory["text"] for memory in memories],
    }
    return {
        "query": query,
        "retrieved_docs": retrieved_docs,
        "retrieved_memories": memories,
        "combined_context": combined_context,
    }
```

```python
def memory_results_to_docs(memories: list[dict[str, Any]]) -> list[dict[str, Any]]:
    pseudo_docs: list[dict[str, Any]] = []
    for index, memory in enumerate(memories, start=1):
        pseudo_docs.append(
            {
                "doc_id": f"memory_{memory['memory_type']}",
                "chunk_id": f"memory_chunk_{index}",
                "text": memory["text"],
                "source": f"memory://{memory['memory_type']}",
                "score": round(float(memory["score"]), 4),
            }
        )
    return pseudo_docs
```

왜 중요한가:
- short/long/vector memory 결과를 한곳에 모아 score 기준으로 다시 정렬한다.
- `memory_results_to_docs()`는 메모리 결과를 문서처럼 바꿔 retriever 결과와 합칠 수 있게 한다.
- 즉, 메모리도 결국 workflow 입장에서는 또 하나의 evidence source가 된다.


In [ ]:
retriever = build_demo_index(persist=False)
retrieved_docs = retriever.search('When does the organization-wide rollout begin?', top_k=3)
context_bundle = build_memory_augmented_context(
    query='How should I answer rollout timing for Mina?',
    retrieved_docs=retrieved_docs,
    short_term_memory=short_term,
    long_term_memory=reloaded_long_term,
    vector_memory=vector_memory,
    top_k=3,
)
{
    'doc_sources': [doc['source'] for doc in context_bundle['retrieved_docs']],
    'memory_count': len(context_bundle['retrieved_memories']),
    'memory_texts': [memory['text'] for memory in context_bundle['retrieved_memories']],
}


### 메모리 업데이트 전략(Memory Update Strategy)

모든 대화를 다 저장하면 메모리 시스템은 금방 오염된다. 감사 인사, 잡담, 일시적인 요청까지 모두 남겨두면 나중에 retrieval이 잡음을 끌고 오기 쉽다. 그래서 메모리 업데이트에는 보통 importance scoring이나 규칙 기반 필터가 필요하다.

이 셀은 후보 문장별 중요도 점수를 먼저 계산한다.

- **목적**: 어떤 문장을 저장할 가치가 있는지 수치로 평가한다.
- **핵심 로직**: `score_memory_importance(text, metadata)`가 텍스트와 메타데이터를 바탕으로 중요도 점수를 계산하고, 이를 표로 보여준다.
- **주요 파라미터/변수**:
  - `candidates`: 저장 후보 문장 목록이다.
  - `importance_score`: 각 후보의 중요도 점수이다.
  - `metadata={'category': 'fact', 'source': 'user'}`: 점수 계산 시 참조하는 힌트 정보이다.

왜 중요한가? "Mina prefers concise rollout summaries..." 같은 문장은 반복 사용될 가능성이 높은 선호 정보지만, "The user said thanks." 같은 문장은 대부분의 경우 영구 저장 가치가 낮다.

### 실제 구현 펼쳐보기: `score_memory_importance()`
```python
def score_memory_importance(text: str, metadata: dict[str, Any] | None = None) -> float:
    normalized = normalize_text(text).lower()
    tokens = content_tokens(normalized)
    metadata = metadata or {}

    score = 0.2
    if len(tokens) >= 8:
        score += 0.2
    if any(marker in normalized for marker in ("remember", "prefers", "prefer", "always", "never", "important")):
        score += 0.25
    if any(character.isdigit() for character in normalized):
        score += 0.1
    if metadata.get("category") in {"preference", "constraint", "fact"}:
        score += 0.15
    if metadata.get("source") == "user":
        score += 0.1
    return round(min(score, 1.0), 3)
```

중요도 점수 기준:
- 길이가 충분한가
- preference/constraint/fact처럼 중요한 category인가
- 숫자나 반복적 표현이 있는가
- user가 직접 말한 내용인가

💡 면접 포인트: 메모리 품질은 retrieval보다 저장 정책에서 먼저 무너지는 경우가 많기 때문에, importance scoring을 명시적으로 두는 것이 중요하다.


In [ ]:
candidates = [
    'Remember: Mina prefers concise rollout summaries with dates first.',
    'The user said thanks.',
    'Important: Team Apollo is piloting the scheduling template in Seoul.',
]
importance_frame = pd.DataFrame(
    {
        'candidate': candidates,
        'importance_score': [score_memory_importance(text, {'category': 'fact', 'source': 'user'}) for text in candidates],
    }
)
importance_frame


이 update 실험은 threshold를 넘는 memory만 저장한다. 핵심은 저장 정책을 inspectable하게 만드는 것이다. 어떤 항목은 남고 어떤 항목은 버려지는지 직접 보아야 memory 설계를 설명할 수 있다.

- **목적**: importance threshold가 실제 저장 여부를 어떻게 가르는지 확인한다.
- **핵심 로직**: `selective_memory_update(...)`가 중요도 점수를 계산한 뒤, `threshold=0.55` 이상이면 long-term/vector memory에 저장하고, 아니면 건너뛴다.
- **주요 파라미터/변수**:
  - `key`: 장기 메모리에 저장할 식별자이다.
  - `threshold=0.55`: 이 값 이상일 때만 저장한다.
  - `category`, `metadata`: 메모리 분류와 출처를 함께 남겨 후속 해석을 돕는다.

예를 들어 아래 코드에서:
- `'mina_style_rule'`: 저장 가치가 높은 선호 규칙이다.
- `'low_signal_event'`: 점수가 낮아 필터링될 가능성이 큰 저신호 이벤트다.

왜 모든 대화를 다 기억하면 안 되는가? 메모리 오염(memory pollution)이 누적되면, agent는 정작 중요한 사실보다 사소한 잡음을 더 자주 꺼내오게 된다.

### 실제 구현 펼쳐보기: `selective_memory_update()`
```python
def selective_memory_update(
    text: str,
    key: str,
    long_term_memory: LongTermMemory | None = None,
    vector_memory: VectorMemory | None = None,
    threshold: float = 0.55,
    category: str = "fact",
    metadata: dict[str, Any] | None = None,
) -> dict[str, Any]:
    metadata = metadata or {}
    effective_metadata = {"category": category, **metadata}
    importance_score = score_memory_importance(text, effective_metadata)
    decision = {
        "key": key,
        "text": normalize_text(text),
        "category": category,
        "importance_score": importance_score,
        "stored": importance_score >= threshold,
        "stored_in": [],
    }
    if importance_score < threshold:
        return decision

    if long_term_memory is not None:
        long_term_memory.store_memory(
            key=key,
            value=text,
            category=category,
            metadata=metadata,
            importance=importance_score,
        )
        decision["stored_in"].append("long_term")

    if vector_memory is not None:
        vector_memory.add_memory(
            text=text,
            metadata={"key": key, "category": category, **metadata},
            importance=importance_score,
        )
        decision["stored_in"].append("vector")

    return decision
```

왜 모든 대화를 다 저장하면 안 되는지 코드로 드러난다.
- threshold 미만이면 바로 return한다.
- threshold 이상일 때만 long-term/vector에 저장한다.
- 어떤 저장소에 들어갔는지도 `stored_in`으로 남긴다.


In [ ]:
update_decisions = [
    selective_memory_update(
        text='Remember: Mina prefers concise rollout summaries with dates first.',
        key='mina_style_rule',
        long_term_memory=reloaded_long_term,
        vector_memory=vector_memory,
        threshold=0.55,
        category='preference',
        metadata={'source': 'user'},
    ),
    selective_memory_update(
        text='The user said thanks.',
        key='low_signal_event',
        long_term_memory=reloaded_long_term,
        vector_memory=vector_memory,
        threshold=0.55,
        category='event',
        metadata={'source': 'user'},
    ),
]
pd.DataFrame(update_decisions)


## Agent workflow 안에서의 memory

이제 메모리를 standalone 실험이 아니라 실제 workflow에 연결해본다. memory-aware workflow는 기본 문서 검색 흐름 위에 memory retrieval과 memory update 단계를 추가한다. 질문을 처리한 뒤, 필요한 기억을 읽고, 최종적으로 새로 배운 내용을 선택적으로 다시 저장하는 구조다.

- **목적**: 메모리 시스템이 실제 workflow state와 trace에 어떻게 연결되는지 확인한다.
- **핵심 로직**: `run_workflow_with_memory(...)`가 기존 workflow를 실행하면서 short-term, long-term, vector memory를 함께 읽고, `update_memory=True`일 때 새 메모리도 기록한다.
- **주요 파라미터/변수**:
  - `include_memories_in_context=False`: 메모리를 context에 직접 넣을지 여부를 제어한다. 실험에서는 retrieval/update 흔적을 먼저 보는 데 집중한다.
  - `update_memory=True`: 실행 후 memory update 단계를 활성화한다.
  - `retrieved_memories`, `memory_updates`: 상태 안에 남은 메모리 읽기/쓰기 결과이다.

이 셀의 출력에서는 `final_answer`뿐 아니라, 몇 개의 memory가 읽혔고 몇 개가 업데이트되었는지를 같이 보자. memory 시스템은 답변 내용만으로 평가하면 놓치는 부분이 많다.

### 실제 구현 펼쳐보기: `run_workflow_with_memory()`
```python
def run_workflow_with_memory(
    query: str,
    retriever: Any,
    short_term_memory: ShortTermMemory | None = None,
    long_term_memory: LongTermMemory | None = None,
    vector_memory: VectorMemory | None = None,
    top_k: int = DEFAULT_TOP_K,
    memory_top_k: int = 3,
    include_memories_in_context: bool = False,
    update_memory: bool = True,
    memory_threshold: float = 0.55,
    trace_path: Path | None = None,
) -> AgentState:
    state = create_initial_state(query)

    try:
        _execute_workflow_steps(
            state,
            MEMORY_WORKFLOW_STEPS,
            retriever=retriever,
            top_k=top_k,
            short_term_memory=short_term_memory,
            long_term_memory=long_term_memory,
            vector_memory=vector_memory,
            memory_top_k=memory_top_k,
            include_memories_in_context=include_memories_in_context,
            update_memory=update_memory,
            memory_threshold=memory_threshold,
        )
    except Exception as error:  # pragma: no cover - defensive path
        record_error(state, str(error))
        error_start = time.perf_counter()
        append_trace(
            state,
            "workflow_error",
            {
                "inputs": {"last_completed_node": state["trace"][-1]["node"] if state["trace"] else None},
                "outputs": {"error": str(error)},
                "latency": round(time.perf_counter() - error_start, 6),
            },
        )
        state["final_answer"] = "Workflow execution failed."
        state["final_status"] = "failed"

    if trace_path is not None:
        write_json(trace_path, state)

    return state
```

기본 workflow와의 차이:
- retrieve_docs만 하지 않고 retrieve_memories를 추가한다.
- 최종적으로 update_memory 단계까지 포함한다.
- 즉, 메모리는 읽기와 쓰기 둘 다 workflow의 명시적 단계가 된다.


In [ ]:
memory_state = run_workflow_with_memory(
    'When does the organization-wide rollout begin?',
    retriever=retriever,
    short_term_memory=short_term,
    long_term_memory=reloaded_long_term,
    vector_memory=vector_memory,
    include_memories_in_context=False,
    update_memory=True,
)
{
    'final_status': memory_state['final_status'],
    'final_answer': memory_state['final_answer'],
    'retrieved_memories': len(memory_state['retrieved_memories']),
    'memory_updates': len(memory_state['memory_updates']),
}


## 시각화(Visualization)

메모리 시스템은 저장소가 여러 개라서, 내부 상태를 시각화하지 않으면 금방 불투명해진다. `display_memory()`는 short-term, long-term, vector memory를 각각 표로 보여주어 현재 어떤 정보가 어디에 들어 있는지 한눈에 확인하게 해준다.

- **목적**: 서로 다른 메모리 저장소의 현재 상태를 동시에 확인한다.
- **핵심 로직**: `display_memory(...)`가 세 종류 메모리를 각각 데이터프레임 또는 요약 표로 렌더링한다.
- **주요 파라미터/변수**:
  - `short_term_memory`: 최근 대화 버퍼이다.
  - `long_term_memory`: JSON 기반 영속 저장소이다.
  - `vector_memory`: 유사도 검색용 메모리 인덱스이다.

디버깅할 때는 "이 정보가 왜 여기 있지?"라는 질문을 자주 하게 된다. 이런 시각화가 있어야 memory 설계가 마법처럼 숨지 않고, 상태 기반 시스템으로 남는다.

### 읽을 때 체크할 점
- 이 셀의 출력은 그 자체보다도 **어느 단계의 가정이 맞았는지/틀렸는지**를 보여주는 신호로 읽는 것이 좋다.
- 스터디나 면접에서는 `무엇을 했다`보다 `왜 이런 단계를 따로 분리했는가`를 설명할 수 있어야 한다.


In [ ]:
display_memory(
    short_term_memory=short_term,
    long_term_memory=reloaded_long_term,
    vector_memory=vector_memory,
)


이 trace view는 memory retrieval과 memory update가 workflow 어디에 들어오는지 보여준다. memory 동작이 "마법처럼" 숨어 있는 것이 아니라, 실행 흔적(trace) 안에 명시적으로 드러난다는 점이 중요하다.

- **목적**: memory 관련 노드가 기본 workflow 단계 사이 어디에 끼어드는지 확인한다.
- **핵심 로직**: `display_trace(memory_state['trace'])`가 전체 trace를 표로 보여주어 `retrieve_memories`, `update_memory` 같은 노드를 눈에 띄게 만든다.
- **주요 파라미터/변수**:
  - `memory_state['trace']`: 문서 retrieval, synthesis, verification뿐 아니라 memory node까지 포함한 실행 기록이다.

trace를 읽을 때는 다음을 확인해보자.
- memory retrieval이 문서 retrieval 이후에 붙는가
- memory update가 workflow 마지막에 수행되는가
- latency가 비정상적으로 큰 메모리 단계는 없는가

이렇게 trace가 있어야 "메모리를 써서 좋아졌다"는 주장도 실제 근거를 갖게 된다.

### 읽을 때 체크할 점
- 이 셀의 출력은 그 자체보다도 **어느 단계의 가정이 맞았는지/틀렸는지**를 보여주는 신호로 읽는 것이 좋다.
- 스터디나 면접에서는 `무엇을 했다`보다 `왜 이런 단계를 따로 분리했는가`를 설명할 수 있어야 한다.


In [ ]:
display_trace(memory_state['trace'])


## 실험

첫 번째 실험은 메모리가 있을 때와 없을 때 context가 어떻게 달라지는지 비교한다. 같은 질문이라도 메모리가 있으면 agent가 사용자 선호나 과거 사실을 함께 참고할 수 있고, 없으면 문서만 보고 답해야 한다.

- **목적**: memory retrieval이 combined context의 내용과 길이를 어떻게 바꾸는지 확인한다.
- **핵심 로직**: `build_memory_augmented_context(...)`를 한 번은 문서만으로, 한 번은 문서+메모리로 호출한 뒤 결과를 표로 비교한다.
- **주요 파라미터/변수**:
  - `no_memory_bundle`: 문서만 포함한 context 결과이다.
  - `with_memory_bundle`: 메모리까지 포함한 context 결과이다.
  - `memory_count`: 실제로 몇 개의 memory가 붙었는지 보여주는 수치이다.

이 표는 memory가 단순히 "더 많이 넣는다"는 뜻이 아니라, 어떤 종류의 추가 문맥을 제공하는지 읽는 데 사용한다. 관련성 높은 선호 정보가 함께 들어오면 답변의 개인화가 쉬워진다.

### 읽을 때 체크할 점
- 이 셀의 출력은 그 자체보다도 **어느 단계의 가정이 맞았는지/틀렸는지**를 보여주는 신호로 읽는 것이 좋다.
- 스터디나 면접에서는 `무엇을 했다`보다 `왜 이런 단계를 따로 분리했는가`를 설명할 수 있어야 한다.


In [ ]:
query = 'How should I answer rollout timing for Mina?'
no_memory_bundle = build_memory_augmented_context(query=query, retrieved_docs=retrieved_docs, top_k=3)
with_memory_bundle = build_memory_augmented_context(
    query=query,
    retrieved_docs=retrieved_docs,
    short_term_memory=short_term,
    long_term_memory=reloaded_long_term,
    vector_memory=vector_memory,
    top_k=3,
)
pd.DataFrame(
    [
        {
            'mode': 'docs_only',
            'memory_count': len(no_memory_bundle['retrieved_memories']),
            'combined_context': str(no_memory_bundle['combined_context']),
        },
        {
            'mode': 'docs_plus_memory',
            'memory_count': len(with_memory_bundle['retrieved_memories']),
            'combined_context': str(with_memory_bundle['combined_context']),
        },
    ]
)


### 실험 2: memory pollution

memory pollution은 저장된 메모리 중 노이즈 비중이 높아져, retrieval 품질이 떨어지는 현상이다. agent 메모리를 설계할 때 가장 흔한 함정 중 하나가 "일단 저장하고 나중에 쓰자" 전략인데, 실제로는 이렇게 저장한 저신호 정보가 관련 기억을 밀어낼 수 있다.

- **목적**: 노이즈 메모리가 vector search 결과를 어떻게 흐릴 수 있는지 확인한다.
- **핵심 로직**: `polluted_memory`에 중요한 사실과 중요하지 않은 잡음을 함께 넣고, 같은 query로 검색했을 때 어떤 항목이 상위에 뜨는지 본다.
- **주요 파라미터/변수**:
  - `polluted_memory`: 의도적으로 잡음을 섞어 만든 실험용 벡터 메모리이다.
  - `metadata={'kind': 'noise'}`: 나중에 노이즈 패턴을 해석하기 위한 레이블이다.

결과를 읽을 때는 중요한 rollout timing 메모리가 여전히 상위에 오는지, 아니면 snack/brand palette 같은 주변 정보가 상단을 차지하는지 보자. 바로 이런 현상을 막기 위해 selective update가 필요하다.

### 읽을 때 체크할 점
- 이 셀의 출력은 그 자체보다도 **어느 단계의 가정이 맞았는지/틀렸는지**를 보여주는 신호로 읽는 것이 좋다.
- 스터디나 면접에서는 `무엇을 했다`보다 `왜 이런 단계를 따로 분리했는가`를 설명할 수 있어야 한다.


In [ ]:
polluted_memory = VectorMemory()
polluted_memory.add_memory('Mina prefers concise rollout answers that begin with the exact launch date.', metadata={'kind': 'preference'}, importance=0.9)
polluted_memory.add_memory('Mina is ordering snacks for the rollout celebration.', metadata={'kind': 'noise'}, importance=0.4)
polluted_memory.add_memory('Rollout posters should use the coral brand palette.', metadata={'kind': 'noise'}, importance=0.4)
polluted_memory.add_memory('The rollout FAQ should mention the May 5, 2025 launch date.', metadata={'kind': 'fact'}, importance=0.8)
pd.DataFrame(polluted_memory.search('How should I answer rollout timing for Mina?', top_k=4))


### 실험 3: memory summarization

short-term memory는 시간이 지나면 길어지므로, 그대로 long-term으로 옮기기보다 요약해 저장하는 전략이 자주 쓰인다. 요약(summarization)은 개별 이벤트를 모두 남기지 않으면서도 중요한 패턴을 보존하는 절충안이다.

- **목적**: 최근 대화 이벤트를 한 줄 요약으로 압축해 long-term memory에 저장하는 흐름을 확인한다.
- **핵심 로직**: `summarize_memory_events(short_term.events, limit=5)`가 최근 이벤트를 요약 텍스트로 만들고, 이를 `store_memory()`로 장기 메모리에 저장한다.
- **주요 파라미터/변수**:
  - `limit=5`: 요약에 반영할 최근 이벤트 수이다.
  - `summary_text`: 생성된 메모리 요약 문자열이다.
  - `recent_memory_summary`: 장기 메모리에 저장할 요약 항목 키이다.

왜 유용한가? 모든 원문 이벤트를 남기면 검색 비용이 커지고 노이즈가 쌓인다. 반대로 너무 aggressive하게 요약하면 중요한 세부가 사라질 수 있다. 이 trade-off가 memory 시스템 설계의 핵심이다.

### 읽을 때 체크할 점
- 이 셀의 출력은 그 자체보다도 **어느 단계의 가정이 맞았는지/틀렸는지**를 보여주는 신호로 읽는 것이 좋다.
- 스터디나 면접에서는 `무엇을 했다`보다 `왜 이런 단계를 따로 분리했는가`를 설명할 수 있어야 한다.


In [ ]:
summary_text = summarize_memory_events(short_term.events, limit=5)
reloaded_long_term.store_memory('recent_memory_summary', summary_text, category='summary')
pd.DataFrame(
    {
        'summary_text': [summary_text],
        'stored_summary': [reloaded_long_term.retrieve_memory('recent_memory_summary')['value']],
    }
)


## 결과 해석

이 마지막 분석 셀은 short-term, long-term, vector memory, workflow integration 관점에서 이번 실험을 한 줄씩 요약한다. 숫자와 관찰을 함께 적어두면, 메모리 실험을 단순 데모가 아니라 설명 가능한 설계 검증으로 바꿀 수 있다.

- **목적**: 세 종류 메모리와 workflow 연결 결과를 요약 표로 정리한다.
- **핵심 로직**: `analysis_frame`에 각 테마별 관찰값을 문자열로 저장해 표로 표시한다.
- **주요 파라미터/변수**:
  - `short_term_memory`: workflow 후 이벤트 개수로 버퍼가 어떻게 변했는지 본다.
  - `persisted items`: long-term memory에 실제로 몇 개가 남았는지 보여준다.
  - `top memory hit`: vector search에서 가장 먼저 검색된 메모리이다.
  - `trace contains memory nodes`: workflow trace 안에 memory node가 실제로 찍혔는지 검증한다.

이 결과를 읽을 때는 "메모리가 있다"보다 "어떤 메모리가 어디에 저장되고 어떻게 검색되었는가"를 중심으로 보자. memory 시스템은 양보다 구조가 더 중요하다.

면접에서는 `세 종류 메모리의 역할이 어떻게 다른지`, `왜 selective update가 필요한지`, `memory results를 왜 다시 docs 형태로 바꾸는지`를 설명할 수 있으면 설계 이해도가 높아 보인다.


In [ ]:
analysis_frame = pd.DataFrame(
    [
        {
            'theme': 'short_term_memory',
            'observation': f"buffer size after workflow run: {len(short_term.events)} events",
        },
        {
            'theme': 'long_term_memory',
            'observation': f"persisted items: {len(reloaded_long_term.list_items())}",
        },
        {
            'theme': 'vector_memory',
            'observation': f"top memory hit: {vector_search_results[0]['text'] if vector_search_results else 'none'}",
        },
        {
            'theme': 'workflow_integration',
            'observation': f"trace contains memory nodes: {any(entry['node'] == 'retrieve_memories' for entry in memory_state['trace'])}",
        },
    ]
)
analysis_frame


## 핵심 정리

이 노트북을 통해 메모리는 하나의 저장소가 아니라, 시간축과 용도에 따라 분리된 여러 계층의 조합이라는 점을 확인했다. short-term memory는 방금 대화한 내용을 붙잡고, long-term memory는 반복적으로 필요한 선호와 사실을 저장하며, vector memory는 의미적으로 비슷한 기억을 찾아온다.

또한 모든 대화를 다 기억하는 것이 좋은 설계가 아니라는 점도 확인했다. importance scoring, selective update, summarization 같은 정책이 없으면 memory pollution이 쉽게 발생하고, 오히려 retrieval 품질을 해칠 수 있다.

💡 면접 포인트: "agent memory는 단순 저장 기능이 아니라, 무엇을 남기고 무엇을 버릴지 결정하는 정책 시스템이다. short-term, long-term, vector memory를 분리하면 개인화와 근거성을 함께 관리하기 쉬워진다"고 설명하면 좋다.

### 읽을 때 체크할 점
- 이 셀의 출력은 그 자체보다도 **어느 단계의 가정이 맞았는지/틀렸는지**를 보여주는 신호로 읽는 것이 좋다.
- 스터디나 면접에서는 `무엇을 했다`보다 `왜 이런 단계를 따로 분리했는가`를 설명할 수 있어야 한다.
